<a href="https://colab.research.google.com/github/inefable12/mecanismos/blob/main/Mecanismo_NEB_XTB_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

$$\Large \textit{Colegio Químico del Perú | Región Cusco}$$
$$\large \textbf{Sesión 6 - Cuaderno 1 | Perfil de Energía}$$

_Dr. Jesus Alvarado Huayhuaz_

1. Relaxed Surface Scan

En español, **Escaneo relajado de la superficie de energía potencial** (también se dice barrido relajado de la superficie de energía potencial).

Idea básica:

Se explora la superficie de energía potencial (PES) variando gradualmente una coordenada geométrica (por ejemplo, una distancia de enlace, ángulo o dihedro) mientras el resto de la estructura se optimiza.

Procedimiento:

Se fija una coordenada estructural (por ejemplo, la distancia entre dos átomos).

Esa coordenada se cambia paso a paso.

En cada paso se optimizan todas las demás coordenadas.

Se obtiene un perfil de energía.

Resultado:

El punto de máxima energía en el perfil suele estar cerca del estado de transición.

Uso típico:

Reacciones simples donde se sabe qué enlace se rompe o se forma.

2. Nudged Elastic Band (NEB)

En español, **Método de la banda elástica nudged** (o más formalmente método de la banda elástica corregida).

Idea básica:

Este método encuentra el camino de mínima energía (MEP, Minimum Energy Path) entre reactivos y productos.

Procedimiento:

Se define la estructura inicial (reactivos) y la final (productos).

Se generan varias estructuras intermedias llamadas imágenes.

Estas imágenes se conectan mediante resortes ficticios (elastic band).

Durante la optimización:

las imágenes se ajustan para seguir el camino de menor energía,

los resortes mantienen una separación uniforme entre ellas.

Resultado:

La imagen con mayor energía corresponde aproximadamente al estado de transición.

Uso típico:

- Reacciones complejas

- Procesos en superficies catalíticas o sólidos

- Cuando no se conoce la coordenada de reacción

| Método               | Traducción                                             | Idea principal                                                    |
| -------------------- | ------------------------------------------------------ | ----------------------------------------------------------------- |
| Relaxed Surface Scan | Escaneo relajado de la superficie de energía potencial | Se modifica una coordenada y se optimiza el resto                 |
| NEB                  | Método de la banda elástica nudged                     | Encuentra el camino de mínima energía entre reactivos y productos |


# 1. Requerimientos

In [ ]:
!pip install condacolab

In [ ]:
import condacolab
condacolab.install()

In [ ]:
!conda install conda-forge::xtb

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 2. Relaxed Surface Scan

## 2.1 Ejemplo: HCN

**HCN  →  HNC** (reacción de isomerización)

Esta es una reacción de isomerización por migración de H. La coordenada de reacción consiste no en romper C–N sino en transferir el hidrógeno.

In [ ]:
#!wget https://raw.githubusercontent.com/inefable12/mecanismos/refs/heads/main/moleculas/R.xyz

Numeración de átomos en en el archivo R.xyz:

1  C

2  N

3  H

La coordenada relevante es:

H–N = 3–2

Las distancias importantes son:

- HCN (Cianuro de hidrógeno o Formonitrilo, 768), llamado R.

H–C ≈ 1.06 Å

H–N ≈ 2.0–2.2 Å

- HNC (Isocianuro de H, 6432654), lo llamaremos P.

H–N ≈ 1.0 Å

## 2.2 Escaneo

In [ ]:
#!rm xtb* charges wbo xcontrol scan.inp # relaxed iterative

In [ ]:
!wget https://raw.githubusercontent.com/inefable12/mecanismos/refs/heads/main/moleculas/scan.inp

In [ ]:
!cat scan.inp # xcontrol

**mode relaxed**

Paso 1

   fijar coordenada

   optimizar geometría

Paso 2

   cambiar coordenada

   optimizar geometría

Paso 3

   cambiar coordenada

   optimizar geometría

**mode iterative**

Paso 1   modificar distancia

Paso 2   modificar distancia

Paso 3   modificar distancia
...

| Característica          | iterative   | relaxed        |
| ----------------------- | ----------- | -------------- |
| Optimización geométrica | ❌ no        | ✅ sí           |
| Precisión del perfil    | baja        | alta           |
| Velocidad               | muy rápida  | más lenta      |
| Uso típico              | exploración | búsqueda de TS |


Inicio 2.2 Å | paso: -0.05 Å | Total de pasos: 24 pasos

rango final:

2.2 − (24 × 0.05) = 1.0 Å

Esto mueve el hidrógeno desde C hacia N.

## 2.3 Comando de ejecución

In [ ]:
!xtb R.xyz --opt verytight --input scan.inp --chrg 0 --uhf 0

# 3. NEB (Nudged Elastic Band)

In [ ]:
!wget https://raw.githubusercontent.com/inefable12/mecanismos/refs/heads/main/moleculas/R.xyz
!wget https://raw.githubusercontent.com/inefable12/mecanismos/refs/heads/main/moleculas/P.xyz

In [ ]:
%%capture
!xtb R.xyz --path P.xyz --gfn2

## 3.1. Leer energías desde xtbpath.xyz

In [ ]:
energies = []

with open("xtbpath.xyz") as f:
    lines = f.readlines()

for line in lines:
    if "energy" in line.lower():
        parts = line.split()
        e = float(parts[1])
        energies.append(e)

energies = np.array(energies)

# convertir a kcal/mol relativas
energies = (energies - energies.min()) * 627.509
energies

## 3.2 Encontrar TS

In [ ]:
ts_index = np.argmax(energies)

print("Número de imágenes:", len(energies))
print("TS aproximado en imagen:", ts_index)
print("Energía de barrera (kcal/mol):", energies[ts_index])

## 3.3 Graficar perfil de energía

In [ ]:
plt.figure()

plt.plot(range(len(energies)), energies/1000, marker="o")

plt.xlabel("Imagen de reacción")
plt.ylabel("Energía relativa (kcal/mol)")
plt.title("Perfil de energía HCN → HNC (xTB-NEB)")

plt.scatter(ts_index, energies[ts_index]/1000, s=100) #, marker="x")

plt.show()